<a href="https://colab.research.google.com/github/gitmystuff/DSChunks/blob/main/Wikinator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikinator


## API

APIs are mechanisms that enable two software components to communicate with each other using a set of definitions and protocols. For example, the weather bureau’s software system contains daily weather data. The weather app on your phone “talks” to this system via APIs and shows you daily weather updates on your phone.

https://aws.amazon.com/what-is/api/

## Wikipedia API

NOTE: If you intend to do any scraping projects or automated requests, consider alternatives such as Pywikipediabot or MediaWiki API, which has other superior features.

* wikipedia.search('keywords', results=2)
* wikipedia.suggest('keyword')
* wikipedia.summary('keywords', sentences=2)
* wikipedia.page('keywords')
* wikipedia.page('keywords').content
* wikipedia.page('keywords').references
* wikipedia.page('keywords').title
* wikipedia.page('keywords').url
* wikipedia.page('keywords').categories
* wikipedia.page('keywords').content
* wikipedia.page('keywords').links
* wikipedia.geosearch(33.2075, 97.1526)
* wikipedia.set_lang('hi')
* wikipedia.languages()
* wikipedia.page('keywords').images[0]
* wikipedia.page('keywords').html()

In [1]:
pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11680 sha256=6e470242fefee66920be9f8396a4c38d2b8d6094a976937d53d4ec7013376986
  Stored in directory: /root/.cache/pip/wheels/5e/b6/c5/93f3dec388ae76edc830cb42901bb0232504dfc0df02fc50de
Successfully built wikipedia


In [3]:
# https://kleiber.me/blog/2017/07/22/tutorial-lda-wikipedia/
import pandas as pd
import random
import wikipedia

# rtitles = wikipedia.random(5)

# get 5 Wikipedia page titles based on keywords
titles = []
keywords = ['ultranationalism', 'religion', '1900s', 'equal rights', 'racism', 'suffrage', 'unemployment']
for key in keywords:
    title = wikipedia.search(key, results=5)
    titles.append(title[0])

# print(titles)
data = []

for title in titles:
    # disambiguous error fix
    try:
        url_title = title.strip().replace(' ', '_')
        url = f'https://en.wikipedia.org/wiki/{url_title}' # left alt, shift, down to duplicate line
        # data.append([title, url, wikipedia.page(title, auto_suggest=False).content, wikipedia.summary(title, auto_suggest=False, sentences=15)])
        data.append([title, url])
    except wikipedia.exceptions.DisambiguationError as e:
        s = random.choice(e.options)
        data.append([title, wikipedia.page(s).content,  wikipedia.summary(title, auto_suggest=False, sentences=15)])

# df = pd.DataFrame(data, columns=['title', 'url', 'content', 'summary'])
pages = pd.DataFrame(data, columns=['title', 'url'])
pages.head()

,title,url
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism
1,Religion,https://en.wikipedia.org/wiki/Religion
2,1900s,https://en.wikipedia.org/wiki/1900s
3,Equal Rights Amendment,https://en.wikipedia.org/wiki/Equal_Rights_Ame...
4,Racism,https://en.wikipedia.org/wiki/Racism


## Beautiful Soup

Beautiful Soup is a Python package for parsing HTML and XML documents, including those with malformed markup. It creates a parse tree for documents that can be used to extract data from HTML, which is useful for web scraping.

https://en.wikipedia.org/wiki/Beautiful_Soup_(HTML_parser)

## Parser

A computer program that breaks down text into recognized strings of characters for further analysis.

https://www.merriam-webster.com/dictionary/parser

In [4]:
# get major headings
from bs4 import BeautifulSoup
import pandas as pd
import requests

data = []

def make_soup(page):
  # global df
  soup = BeautifulSoup(requests.get(page.url).text)
  s = soup.find_all('h2')
  s_list = [x.get_text().replace('[edit]', '') for x in s]
  # print(pd.Series(s_list))
  data.extend([[page.title, page.url, x.get_text().replace('[edit]', '')] for x in s])

x = pages.apply(make_soup, axis=1)
headings = pd.DataFrame(data, columns=['title', 'url', 'heading'])
drop_list = ['Contents', 'See also', 'References', 'External links', 'Notes', 'Sources', 'Further reading', 'Bibliography']
headings = headings[~headings['heading'].isin(drop_list)]
print(headings.shape)
headings.head()

(58, 3)


,title,url,heading
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist organizations
5,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist terrorism


In [5]:
headings['title'].value_counts()

title
Equal Rights Amendment    14
Racism                    12
1900s                      9
Religion                   8
Ultranationalism           6
Unemployment               5
Suffrage                   4
Name: count, dtype: int64

In [6]:
# get subheadings and text
import re

CLEANR = re.compile('<.*?>')
def cleanhtml(raw_html):
  cleantext = re.sub(CLEANR, '', raw_html)
  return cleantext

data = []
def get_subs(row):
  heading1 = row['heading']
  title = row['title']
  url = row['url']
  soup = BeautifulSoup(requests.get(url).text)
  txt = ''
  txt1 = ''
  target = soup.find('span', attrs={'id': heading1.replace(' ', '_')}).parent
  for sib in target.find_next_siblings():
      if sib.name=='h2':
          break
      else:
          txt += str(sib)
          if sib.name=='p':
            txt1 += str(sib)

  soup2 = BeautifulSoup(txt)
  s = soup2.find_all('h3')
  s_list2 = [x.get_text().replace('[edit]', '') for x in s]
  # print(f'{heading1}\n')
  if len(s_list2) > 0:
    # print(pd.Series(s_list2))
    for i in range(len(s_list2)):
      txt=''
      heading2 = s_list2[i]
      target2 = soup.find('h3', string=heading2)
      target2 = soup.find('span', attrs={'id': heading2.replace(' ', '_')}).parent
      for sib in target2.find_next_siblings():
          if sib.name=='h3':
              break
          else:
            if sib.name=='p':
              txt += sib.text

      data.append([title, url, heading1, heading2, cleanhtml(txt)])
  else:
      data.append([title, url, heading1, 'None', cleanhtml(txt1)])

x = headings.apply(get_subs, axis=1)
df = pd.DataFrame(data, columns=['title', 'url', 'heading', 'subheading', 'txt'])
print(df.shape)
df.head()

(184, 5)


,title,url,heading,subheading,txt
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,None,British political theorist Roger Griffin has s...
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,None,American historian Walter Skya has written in ...
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Currently represented in national governments ...,The following political parties have been char...
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Represented parties with former ultranationali...,The following political parties historically h...
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Formerly represented in national governments o...,Arising out of strident Sri Lankan Tamil natio...


## Natural Language Processing

https://github.com/gitmystuff/DSChunks/blob/main/Natural%20Language%20Processing.ipynb

## LDA (Latent Dirichlet Allocation)

In natural language processing, Latent Dirichlet Allocation (LDA) is a Bayesian network (and, therefore, a generative statistical model) for modeling automatically extracted topics in textual corpora. The LDA is an example of a Bayesian topic model. In this, observations (e.g., words) are collected into documents, and each word's presence is attributable to one of the document's topics. Each document will contain a small number of topics.

Sources:
 * https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation
 * https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html

## TFIDF Vectorizer

TF-IDF will transform the text into meaningful representation of integers or numbers which is used to fit machine learning algorithm for predictions. TF-IDF Vectorizer is a measure of originality of a word by comparing the number of times a word appears in document with the number of documents the word appears in.

https://www.projectpro.io/recipes/use-tf-df-vectorizer

In [7]:
# identify topics
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer

results = 10
components = 10
topics = 10

vectorizer = TfidfVectorizer(stop_words='english')
vectors = vectorizer.fit_transform(df['txt'].values.astype('U'))

model = LatentDirichletAllocation(n_components=components)
model.fit(vectors)

topics_dictionary = {}
for index, topic in enumerate(model.components_):
    print(f'Topic {index} top words: {[vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-topics:]]}')
    topics_dictionary[index] = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-topics:]]



Topic 0 top words: ['religions', 'states', 'racial', 'race', 'women', 'vote', 'racism', 'era', 'religion', 'unemployment']
Topic 1 top words: ['curve', 'requirements', 'thirty', 'dominant', 'practise', 'votes', 'generation', 'property', 'characterised', 'members']
Topic 2 top words: ['psychology', 'inherently', 'identify', 'superior', 'dharma', 'supremacy', 'governor', 'segregation', 'wealth', 'recall']
Topic 3 top words: ['cost', 'targeted', 'prominent', 'worth', 'assassination', 'stamp', 'postage', 'killings', 'assassinations', 'cent']
Topic 4 top words: ['constitutions', '19th', 'homo', 'international', 'belief', 'business', 'provisions', 'city', 'science', 'theories']
Topic 5 top words: ['functional', 'council', 'belief', 'constituencies', 'atheistic', 'aversive', 'gods', 'discourse', 'refers', 'superstition']
Topic 6 top words: ['citizen', 'interpreted', 'naturalized', 'passive', 'derogating', 'cyclical', 'structural', 'electoral', 'council', 'schlafly']
Topic 7 top words: ['conve

In [8]:
def get_topics(row):
  return ', '.join([top for top in topics_dictionary[row.topic_idx]])

topic_results = model.transform(vectors)
df['topic_idx'] = topic_results.argmax(axis=1)

df['topics']= df.apply(get_topics, axis=1)
df.head()

,title,url,heading,subheading,txt,topic_idx,topics
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,None,British political theorist Roger Griffin has s...,7,"converts, 1900, estimates, vary, billion, 710,..."
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,None,American historian Walter Skya has written in ...,4,"constitutions, 19th, homo, international, beli..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Currently represented in national governments ...,The following political parties have been char...,0,"religions, states, racial, race, women, vote, ..."
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Represented parties with former ultranationali...,The following political parties historically h...,0,"religions, states, racial, race, women, vote, ..."
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Formerly represented in national governments o...,Arising out of strident Sri Lankan Tamil natio...,8,"ethnocentric, british, suffragium, 109, 1903, ..."


## SpaCy

spaCy is an open-source software library for advanced natural language processing, written in the programming languages Python and Cython.

* https://en.wikipedia.org/wiki/SpaCy
* https://spacy.io/
* https://medium.com/analytics-vidhya/text-summarization-using-spacy-ca4867c6b744

## Language Model and Pipelines

en_core_web_sm

en_core_web_sm is a small English pipeline trained on written web text (blogs, news, comments), that includes vocabulary, syntax and entities.

* https://www.kdnuggets.com/2021/03/natural-language-processing-pipelines-explained.html
* https://spacy.io/usage/spacy-101
* https://en.wikipedia.org/wiki/Language_model
* https://builtin.com/data-science/beginners-guide-language-models

## Pipeline

A language model pipeline is a series of connected steps that transform raw text data into a desired output for further analysis or use. It's similar to a factory assembly line, where each step refines the material until it reaches its final form.

Gemini, July 2024

In [9]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
from collections import Counter
from heapq import nlargest

nlp = spacy.load('en_core_web_sm')

In [10]:
# get example text
import textwrap

textwrap.fill(df.iloc[0]['txt'])

'British political theorist Roger Griffin has stated that\nultranationalism is essentially founded on xenophobia in a way that\nfinds supposed legitimacy "through deeply mythicized narratives of\npast cultural or political periods of historical greatness or of old\nscores to settle against alleged enemies". It can also draw on\n"vulgarized forms" of different aspects of the natural sciences such\nas anthropology and genetics, eugenics specifically playing a role, in\norder "to rationalize ideas of national superiority and destiny, of\ndegeneracy and subhumanness" in Griffin\'s opinion. Ultranationalists\nview the modern nation-state as, according to Griffin, a living\norganism directly akin to a physical person such that it can decay,\ngrow, die, and additionally experience rebirth. He has highlighted\nNazi Germany as a regime which was founded on ultranationalism.[3]\nUltranationalist activism can adopt varying attitudes towards\nhistorical traditions within the populace. For instance

In [11]:
# get sentences by importance scores
import textwrap
import re

# data = []
# summary_text = ' '.join([re.sub("\[.*?\]", "", txt) for txt in df.txt])
# doc = nlp(summary_text)
summary_text = ' '.join([re.sub("\[.*?\]", "", df.iloc[0]['txt'])])
doc = nlp(summary_text)
keyword = []
stopwords = list(STOP_WORDS)
pos_tag = ['PROPN', 'ADJ', 'NOUN', 'VERB']
for token in doc:
    if(token.text in stopwords or token.text in punctuation):
        continue
    if(token.pos_ in pos_tag):
        keyword.append(token.text)

freq_word = Counter(keyword)
max_freq = Counter(keyword).most_common(1)[0][1]
for word in freq_word.keys():
    freq_word[word] = (freq_word[word]/max_freq)

sent_strength={}
for sent in doc.sents:
    for word in sent:
        if word.text in freq_word.keys():
            if sent in sent_strength.keys():
                sent_strength[sent] += freq_word[word.text]
            else:
                sent_strength[sent] = freq_word[word.text]

    try:
      data.append([sent_strength[sent], str(sent)])
    except:
      pass
    print(sent_strength[sent])
    print(textwrap.fill(str(sent)))
    print()

# summary = nlargest(10, sent_strength, key=sent_strength.get)
# summary = ' '.join([w.text for w in summary])
# print(textwrap.fill(summary, 100))
# df2 = pd.DataFrame(data, columns=['strength', 'txt'])
# df2.sort_values(by=['strength'], ascending=False).head()

12.0
British political theorist Roger Griffin has stated that
ultranationalism is essentially founded on xenophobia in a way that
finds supposed legitimacy "through deeply mythicized narratives of
past cultural or political periods of historical greatness or of old
scores to settle against alleged enemies".

8.999999999999998
It can also draw on "vulgarized forms" of different aspects of the
natural sciences such as anthropology and genetics, eugenics
specifically playing a role, in order "to rationalize ideas of
national superiority and destiny, of degeneracy and subhumanness" in
Griffin's opinion.

6.999999999999997
Ultranationalists view the modern nation-state as, according to
Griffin, a living organism directly akin to a physical person such
that it can decay, grow, die, and additionally experience rebirth.

2.6666666666666665
He has highlighted Nazi Germany as a regime which was founded on
ultranationalism.

3.0
Ultranationalist activism can adopt varying attitudes towards
histor

In [ ]:
len(sent_strength)

11

In [12]:
summary = nlargest(int(len(sent_strength)/2), sent_strength, key=sent_strength.get)
summary = ' '.join([w.text for w in summary])
summary = ' '.join([re.sub("\[.*?\]", "", summary)])
print(textwrap.fill(summary))

According to American scholar Janusz Bugajski, summing up the doctrine
in practical terms, "in its most extreme or developed forms, ultra-
nationalism resembles fascism, marked by a xenophobic disdain of other
nations, support for authoritarian political arrangements verging on
totalitarianism, and a mythical emphasis on the 'organic unity'
between a charismatic leader, an organizationally amorphous movement-
type party, and the nation". British political theorist Roger Griffin
has stated that ultranationalism is essentially founded on xenophobia
in a way that finds supposed legitimacy "through deeply mythicized
narratives of past cultural or political periods of historical
greatness or of old scores to settle against alleged enemies". It can
also draw on "vulgarized forms" of different aspects of the natural
sciences such as anthropology and genetics, eugenics specifically
playing a role, in order "to rationalize ideas of national superiority
and destiny, of degeneracy and subhumannes

In [13]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation

nlp = spacy.blank('en')
nlp.add_pipe('sentencizer')

# https://www.educative.io/answers/text-summarization-in-spacy-and-nltk
# df.iloc[0]['txt']
def summarizer(row):
  txt = row['txt']
  text = ' '.join([re.sub('\[.*?\]|"', '', txt)])
  doc = nlp(text)

  word_frequencies = {}
  for token in doc:
      if token.text not in STOP_WORDS and token.text not in punctuation:
          if token.text not in word_frequencies:
              word_frequencies[token.text] = 1
          else:
              word_frequencies[token.text] += 1


  sorted_sentences = sorted(doc.sents, key=lambda sent: sum(word_frequencies[token.text]
                          for token in sent if token.text in word_frequencies), reverse=True)

  # return str(' '.join(sent.text for sent in sorted_sentences[:int(len(sorted_sentences)/4)]).strip())
  return str(' '.join(sent.text for sent in sorted_sentences[:2]).strip())

# print(textwrap.fill(summarizer(df.iloc[0]['txt'])))

In [14]:
df['summary']= df.apply(summarizer, axis=1)
df.head()

,title,url,heading,subheading,txt,topic_idx,topics,summary
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,None,British political theorist Roger Griffin has s...,7,"converts, 1900, estimates, vary, billion, 710,...","According to American scholar Janusz Bugajski,..."
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,None,American historian Walter Skya has written in ...,4,"constitutions, 19th, homo, international, beli...","In late 2015, the Israeli political journalist..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Currently represented in national governments ...,The following political parties have been char...,0,"religions, states, racial, race, women, vote, ...",The following political parties have been desc...
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Represented parties with former ultranationali...,The following political parties historically h...,0,"religions, states, racial, race, women, vote, ...",The following political parties have historica...
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Formerly represented in national governments o...,Arising out of strident Sri Lankan Tamil natio...,8,"ethnocentric, british, suffragium, 109, 1903, ...",The assassination of Pavlos Fyssas in Septembe...


In [15]:
# example summary
print(textwrap.fill(df.iloc[4].summary))

The assassination of Pavlos Fyssas in September 2013, a hip-hop
musician with left-wing views, from stabbing wounds to the heart and
ribs that occurred after his surrounding by multiple dozen Golden Dawn
militants triggered widespread outrage at the Greek political
organization.  The video game Call of Duty 4: Modern Warfare has
gained notice for its depiction of a civil war inside Russia between
the country's government and an ultranationalist faction, with the
entertainment production being released in 2007.


In [16]:
# example of using summarizer
print(textwrap.fill(summarizer(df.iloc[0])))

According to American scholar Janusz Bugajski, summing up the doctrine
in practical terms, in its most extreme or developed forms, ultra-
nationalism resembles fascism, marked by a xenophobic disdain of other
nations, support for authoritarian political arrangements verging on
totalitarianism, and a mythical emphasis on the 'organic unity'
between a charismatic leader, an organizationally amorphous movement-
type party, and the nation. British political theorist Roger Griffin
has stated that ultranationalism is essentially founded on xenophobia
in a way that finds supposed legitimacy through deeply mythicized
narratives of past cultural or political periods of historical
greatness or of old scores to settle against alleged enemies.


## NER

In [17]:
# uncomment to download
import spacy.cli

spacy.cli.download('en_core_web_sm')

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [18]:
# create ner

nlp = spacy.load('en_core_web_sm')

def ner(row):
  global nlp
  txt = row['txt']
  text = ' '.join([re.sub('\[.*?\]|"', '', txt)])
  doc = nlp(text)
  for ent in doc.ents:
      # print(ent.text, ent.label_, str(spacy.explain(ent.label_)))
      return ent.text, ent.label_, str(spacy.explain(ent.label_))


df['ner']= df.apply(ner, axis=1)
df['ner'].head()

0    (British, NORP, Nationalities or religious or ...
1    (American, NORP, Nationalities or religious or...
2                                                 None
3                                                 None
4    (Sri Lankan Tamil, PERSON, People, including f...
Name: ner, dtype: object

## Text Emdbedding

A text embedding is a piece of text projected into a high-dimensional latent space. The position of our text in this space is a vector, a long sequence of numbers. Think of the two-dimensional cartesian coordinates from algebra class, but with more dimensions—often 768 or 1536.

* https://stackoverflow.blog/2023/11/09/an-intuitive-introduction-to-text-embeddings/#:~:text=A%20text%20embedding%20is%20a,dimensions%E2%80%94often%20768%20or%201536.
* https://huggingface.co/blog/getting-started-with-embeddings

In [20]:
from google.colab import userdata

model_id = 'sentence-transformers/all-MiniLM-L6-v2'

api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/{model_id}'
headers = {"Authorization": f"Bearer {userdata.get('HF_TOKEN')}"}

def get_embeddings(row):
  texts = str(row.summary)
  response = requests.post(api_embedding, headers=headers, json={"inputs": texts, "options":{"wait_for_model":True}})
  return response.json()


In [21]:
df['embeddings']= df.apply(get_embeddings, axis=1)
print(df.shape)
print(df.info())
df.head()

(184, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   title       184 non-null    object
 1   url         184 non-null    object
 2   heading     184 non-null    object
 3   subheading  184 non-null    object
 4   txt         184 non-null    object
 5   topic_idx   184 non-null    int64 
 6   topics      184 non-null    object
 7   summary     184 non-null    object
 8   ner         135 non-null    object
 9   embeddings  184 non-null    object
dtypes: int64(1), object(9)
memory usage: 14.5+ KB
None


,title,url,heading,subheading,txt,topic_idx,topics,summary,ner,embeddings
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,None,British political theorist Roger Griffin has s...,7,"converts, 1900, estimates, vary, billion, 710,...","According to American scholar Janusz Bugajski,...","(British, NORP, Nationalities or religious or ...","[0.053972408175468445, -0.05422463268041611, -..."
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,None,American historian Walter Skya has written in ...,4,"constitutions, 19th, homo, international, beli...","In late 2015, the Israeli political journalist...","(American, NORP, Nationalities or religious or...","[-0.008491200394928455, -0.010670002549886703,..."
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Currently represented in national governments ...,The following political parties have been char...,0,"religions, states, racial, race, women, vote, ...",The following political parties have been desc...,None,"[0.04193244129419327, -0.1433926522731781, -0...."
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Represented parties with former ultranationali...,The following political parties historically h...,0,"religions, states, racial, race, women, vote, ...",The following political parties have historica...,None,"[0.037506088614463806, -0.13643701374530792, -..."
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Formerly represented in national governments o...,Arising out of strident Sri Lankan Tamil natio...,8,"ethnocentric, british, suffragium, 109, 1903, ...",The assassination of Pavlos Fyssas in Septembe...,"(Sri Lankan Tamil, PERSON, People, including f...","[-0.014400621876120567, -0.004642337094992399,..."


In [ ]:
# length of embedding
len(df['embeddings'].iloc[0])

384

## Vector Similarity

$cos(\theta) = A*B/||A||*||B||$

* L2 norms:
* dot product: $a*b = \sum{a_ib_i}$

Readings:
* https://community.openai.com/t/understanding-text-embedding-ada-002-vector-length-of-1536/464737
* https://www.pinecone.io/learn/vector-similarity/


In [34]:
import requests
import numpy as np
from transformers import pipeline
from google.colab import userdata
import textwrap

gen = pipeline("text-generation")

model_id = 'sentence-transformers/all-MiniLM-L6-v2'

api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/{model_id}'
api_gen = 'https://api-inference.huggingface.co/models/gpt2'
headers = {"Authorization": f"Bearer {userdata.get('HF_TOKEN')}"}

def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

def vector_similarity(vec1, vec2):
  try:
    return np.dot(np.array(vec1), np.array(vec2))
  except:
    pass

def query(payload):
    response = requests.post(api_gen, headers=headers, json=payload)
    return response.json()

def embed(texts):
    response = requests.post(api_embedding, headers=headers, json={"inputs": texts, "options":{"wait_for_model":True}})
    return response.json()

def ask_me_something():
  # question = input('Ask me something: ')
  question = 'What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies in the early 1900s?'
  embedded_prompt = embed(question)
  df['similarity'] = df['embeddings'].apply(lambda vector: vector_similarity(vector, embedded_prompt))
  context = ' '.join(df.nlargest(3, 'similarity')['summary'])
  prompt = f'''
    Only answer the question below if you have 100% certainty of the facts. Answer the question in English.
    Context: {context}
    Q: {question}
    A:
    '''

  out = gen(prompt, max_length=1024)
  print(wrap(out[0]['generated_text']))

ask_me_something()


No model was supplied, defaulted to openai-community/gpt2 and revision 6c0e608 (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



    Only answer the question below if you have 100% certainty of the
facts.  Answer the question in English.
    Context: For instance,
India is still one of the most religious countries and religion still
has a strong impact on politics, given that Hindu nationalists have
been targeting minorities like the Muslims and the Christians, who
historically belonged to the lower castes.  By contrast, countries
such as China or Japan are largely secular and thus religion has a
much smaller impact on politics.  In the field of comparative
religion, a common geographical classification of the main world
religions includes Middle Eastern religions (including Zoroastrianism
and Iranian religions), Indian religions, East Asian religions,
African religions, American religions, Oceanic religions, and
classical Hellenistic religions.  Some academics studying the subject
have divided religions into three broad categories:
Some recent
scholarship has argued that not all types of religion are necessari

In [ ]:
df.head()

,title,url,heading,subheading,txt,topic_idx,topics,summary,ner,embeddings,similarity
0,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Background concepts and broader context,None,British political theorist Roger Griffin has s...,2,"ultranationalists, 1945, living, theories, app...","According to American scholar Janusz Bugajski,...","(British, NORP, Nationalities or religious or ...","[0.053972408175468445, -0.05422463268041611, -...",0.480611
1,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Historical movements and analysis,None,American historian Walter Skya has written in ...,1,"converts, state, mongol, putin, film, ethnic, ...","In late 2015, the Israeli political journalist...","(American, NORP, Nationalities or religious or...","[-0.008491200394928455, -0.010670002549886703,...",0.365512
2,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Currently represented in national governments ...,The following political parties have been char...,9,"ultranationalist, world, religious, war, italy...",The following political parties have been desc...,None,"[0.04193244129419327, -0.1433926522731781, -0....",0.487045
3,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Represented parties with former ultranationali...,The following political parties historically h...,9,"ultranationalist, world, religious, war, italy...",The following political parties have historica...,None,"[0.037506088614463806, -0.13643701374530792, -...",0.467666
4,Ultranationalism,https://en.wikipedia.org/wiki/Ultranationalism,Ultranationalist political parties,Formerly represented in national governments o...,Arising out of strident Sri Lankan Tamil natio...,1,"converts, state, mongol, putin, film, ethnic, ...",The assassination of Pavlos Fyssas in Septembe...,"(Sri Lankan Tamil, PERSON, People, including f...","[-0.014400621876120567, -0.004642337094992399,...",0.064484


What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies

In [29]:
import requests
import numpy as np
from transformers import pipeline
from google.colab import userdata
import textwrap

api_embedding = f'https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/all-MiniLM-L6-v2'
api_gen = 'https://api-inference.huggingface.co/models/openai-community/gpt2'
headers = {"Authorization": f"Bearer {userdata.get('HF_TOKEN')}"}

def wrap(x):
  return textwrap.fill(x, replace_whitespace=False, fix_sentence_endings=True)

def vector_similarity(vec1, vec2):
  try:
    return np.dot(np.array(vec1), np.array(vec2))
  except:
    pass

def query(payload):
    response = requests.post(api_gen, headers=headers, json=payload)
    return response.json()

def embed(texts):
    response = requests.post(api_embedding, headers=headers, json={"inputs": texts, "options":{"wait_for_model":True}})
    return response.json()

def ask_me_something_else():
  # question = input('Ask me something: ')
  question = 'What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies in the early 1900s?'
  embedded_prompt = embed(question)
  df['similarity'] = df['embeddings'].apply(lambda vector: vector_similarity(vector, embedded_prompt))
  context = ' '.join(df.nlargest(2, 'similarity')['summary'])

  prompt = f'''
    Only answer the question below if you have 100% certainty of the facts.
    Context: {context}
    Q: {question}
    A:
    '''

  data = query(
      {
          "inputs": prompt
      }
  )

  print(wrap(data[0]['generated_text']))

ask_me_something_else()



    Only answer the question below if you have 100% certainty of the
facts.
    Context: For instance, India is still one of the most
religious countries and religion still has a strong impact on
politics, given that Hindu nationalists have been targeting minorities
like the Muslims and the Christians, who historically belonged to the
lower castes.  By contrast, countries such as China or Japan are
largely secular and thus religion has a much smaller impact on
politics.  In the field of comparative religion, a common geographical
classification of the main world religions includes Middle Eastern
religions (including Zoroastrianism and Iranian religions), Indian
religions, East Asian religions, African religions, American
religions, Oceanic religions, and classical Hellenistic religions.
Some academics studying the subject have divided religions into three
broad categories:
Some recent scholarship has argued that not all
types of religion are necessarily separated by mutually exclusive

What is the connection between ultranationalism and religion and what countries have implemented ultranational and religious policies